# Homework: three charts, worked example

One version of the homework, to compare against yours. Yours should ask
different questions.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

here = Path.cwd()
while not (here / "data" / "messy").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)


def clean_plays(messy):
    """Return a cleaned copy of the messy listening log.

    Decisions made here, on purpose:
      * rows with no duration are dropped, because they cannot answer any
        question about minutes, and there are only 94 of them
      * rows with no device, genre or country are kept and labelled
        "unknown", because they are still real plays
      * dates are parsed per format, because one dayfirst pass misreads every
        ISO date in the file
    """
    df = messy.rename(columns={"minutes played ": "minutes_played"}).copy()

    for column in ["artist_name", "track_name", "genre", "country", "device"]:
        df[column] = df[column].str.strip()
    df["genre"] = df["genre"].str.title()
    df["device"] = df["device"].str.lower()

    df["minutes_played"] = pd.to_numeric(
        df["minutes_played"].str.replace(" min", "", regex=False)
                            .str.replace(" ", "", regex=False),
        errors="coerce",
    )

    raw = df["played_at"].str.strip()
    df["played_at"] = (
        pd.to_datetime(raw, format="%Y-%m-%d", errors="coerce")
        .fillna(pd.to_datetime(raw, format="%d/%m/%Y", errors="coerce"))
        .fillna(pd.to_datetime(raw, format="%d-%m-%Y", errors="coerce"))
    )

    df = df.drop_duplicates().dropna(subset=["minutes_played"])
    df["device"] = df["device"].fillna("unknown")
    df["genre"] = df["genre"].fillna("Unknown")
    df["country"] = df["country"].fillna("Unknown")
    return df.reset_index(drop=True)


raw_messy = pd.read_csv("data/messy/plays_messy.csv")
plays = clean_plays(raw_messy)

lost = len(raw_messy) - len(plays)
print(f"{raw_messy.shape} -> {plays.shape}")
print(f"{lost} rows removed ({lost / len(raw_messy) * 100:.1f}%)")

## Chart 1, line: did the two years differ?

**Question:** is the seasonal pattern the same in 2024 and 2025, or did the
second year just have less listening overall?

In [ ]:
monthly = plays.set_index("played_at")["minutes_played"].resample("MS").sum()
by_year = {year: monthly[monthly.index.year == year] for year in (2024, 2025)}

fig, ax = plt.subplots(figsize=(9, 4.5))
for year, series in by_year.items():
    ax.plot(series.index.month, series.values, marker="o", label=str(year))

ax.set_title("The summer dip repeats, and 2025 was quieter all year")
ax.set_xlabel("Month")
ax.set_ylabel("Minutes played")
ax.set_xticks(range(1, 13))
ax.set_ylim(bottom=0)
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT / "hw_minutes_by_month_by_year.png", dpi=150)
plt.show()

for year, series in by_year.items():
    print(f"{year}: {series.sum():,.0f} minutes over {len(series)} months")

## Chart 2, bar: which devices get skipped most?

**Question:** is there a real difference in skip rate between devices?

In [ ]:
skip_rate = (plays.groupby("device")["skipped"].mean() * 100).sort_values(ascending=False)
counts = plays["device"].value_counts()

fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(skip_rate.index, skip_rate.values, color="#4f6ddb")

ax.set_title("Skip rates differ by device, and 'unknown' tops the list")
ax.set_xlabel("Device")
ax.set_ylabel("Plays skipped (%)")
ax.set_ylim(bottom=0)
ax.grid(axis="y", alpha=0.3)

# Put the number of plays on each bar, so the reader can judge the evidence.
for bar, device in zip(bars, skip_rate.index):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
            f"n={counts[device]}", ha="center", fontsize=9, color="#7d8ba0")

fig.tight_layout()
fig.savefig(OUTPUT / "hw_skip_rate_by_device.png", dpi=150)
plt.show()

print(skip_rate.round(1))

Putting `n=` on each bar is worth the three extra lines. It is the difference
between "the speaker is skipped most" and "the speaker is skipped most, on
314 plays", and a reader can only judge the second one.

Two things worth noticing in that chart.

**`unknown` has the highest skip rate of all**, which is not a device. Either
that is a coincidence on 65 plays, or the device was not recorded *because*
the play was abandoned quickly. Interesting either way, and not something I
went looking for.

**The `unknown` bar is easy to misread** as a kind of device. That is the
cost of my decision to label missing values rather than drop them, and it is
why the decision belongs in the write-up.

## Chart 3, histogram: how long is a play?

**Question:** what does a typical play look like, and do the skips explain
the short ones?

In [ ]:
kept = plays[plays["skipped"] == 0]["minutes_played"]
skipped = plays[plays["skipped"] == 1]["minutes_played"]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist([kept, skipped], bins=25, stacked=True,
        color=["#4f6ddb", "#ff7085"], label=["played through", "skipped"])

ax.set_title("Nearly every skip is under two minutes")
ax.set_xlabel("Minutes played")
ax.set_ylabel("Number of plays")
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT / "hw_play_length.png", dpi=150)
plt.show()

print(f"skipped: {len(skipped)} plays, average {skipped.mean():.2f} minutes")
print(f"kept:    {len(kept)} plays, average {kept.mean():.2f} minutes")
under_two = plays[plays["minutes_played"] < 2]
print(f"share of skips under two minutes:  {(skipped < 2).mean() * 100:.0f}%")
print(f"share of sub-two-minute plays that were skips: "
      f"{under_two['skipped'].mean() * 100:.0f}%")

---

# findings.md

This is what the written half should look like. It is in
`sessions/session-09/homework/solutions/findings.md` as a file too.

## Chart 1, minutes per month

Both years show the same shape: a dip through June, July and August, and the
heaviest listening in the autumn and midwinter. The summer months average
about 245 minutes against about 397 for the rest of the year, so the effect
is large rather than marginal. 2025 came to 4,187 minutes against 4,421 in
2024, so it was about 5% quieter overall.

**What this does not show:** anything about why. A holiday, a change of job,
or a different way of listening that this log does not capture would all
produce the same picture. It also cannot tell me whether 2025 being lower is
a trend or two years of noise, because two points is not a trend.

## Chart 2, skip rate by device

Ignoring the `unknown` group, the speaker has the highest skip rate at 14.0%
and the car the lowest at 9.3%. The ordering is plausible: skipping a track
in the car means reaching for a phone while driving.

`unknown` actually tops the list at 15.4%. That may be a coincidence on 65
plays, or the device may go unrecorded precisely when a play is abandoned
quickly. I do not know, and the chart cannot tell me.

**What this does not show:** that any of these differences are real. The gap
between the top and bottom device is about five percentage points, on group
sizes from 65 to 964 plays, so ordinary luck could produce it. I would want
several times this much data before telling anyone the car matters.

## Chart 3, play length

Play lengths cluster between about three and six minutes, with a sharp spike
of very short plays. 98% of skips last under two minutes, and the longest
skip in the whole dataset is 2.67 minutes.

**What this does not show, and I nearly wrote it anyway:** that short plays
are skips. Only 64% of the plays under two minutes were skipped. "Nearly
every skip is short" and "nearly every short play is a skip" are different
claims, and this chart only supports the first. I had the title the wrong way
round in my first draft, which is a good argument for checking the number
behind every sentence you write.

## Cleaning decisions, and what they cost

I removed 134 of 2,223 rows, which is 6.0% of the file. Of those, 40 were
exact duplicates, identical down to the `play_id`, which is supposed to be
unique, so removing them is safe. The other 94 had no duration recorded. I
dropped them because every question I asked is about minutes, and a play with
no minutes cannot answer any of them.

The alternative was to fill those 94 with the average, which keeps every row
and puts the total at 8,995 minutes instead of 8,608. I did not, because
the totals in chart 1 would then partly be numbers I invented. For a
question about *counts* rather than minutes I would have kept them.

I kept the rows with a missing device, genre or country and labelled them
`unknown`, since they are still real plays. That does mean chart 2 has an
`unknown` bar covering 65 plays, which is honest but easy to misread as a
kind of device, and it happens to have the highest skip rate on the chart.

Dates arrived in three formats. I parsed each one explicitly rather than
using `pd.to_datetime(..., dayfirst=True)`, which runs without complaint and
misreads all 1,793 ISO dates. Every monthly chart above would have been wrong
and nothing would have warned me.